# fact_6 · Fonte를 웹사이트에 연결하는 임시 서버 (캐글, GPU 2개)
캐글 노트북에서 Fonte(Gemma 4 E4B + LoRA `fonte`)를 켜고, 임시 https 주소를 만들어요. 사이트 관리자 페이지에 **주소와 비밀번호**를 넣으면 LAGO 사이트의 Fonte 모드가 이 모델로 답해요.

**GPU 2개 사용 방식:** GPU마다 모델을 하나씩 올려서(일꾼 2개), 질문이 동시에 오면 놀고 있는 쪽이 맡아요. → 한 질문의 답이 빨라지는 게 아니라 **동시에 처리하는 수가 2배**예요. (4B 모델은 GPU 하나에 들어가서, 한 답을 둘이 나눠 만드는 방식은 오히려 느려져요.)

**설정:** GPU **T4 x2** · Internet **On** · Add-ons → Secrets에 `HF_TOKEN` 체크 (`MODEL_KEY`를 Secrets에 넣어 두면 매번 같은 비밀번호를 써요)

**주의**
- 이 노트북이 켜져 있는 동안만 작동해요 (캐글 세션 최대 9~12시간). 다시 켜면 주소가 바뀌어요 → 관리자에 다시 입력.
- `MODEL_NAME`은 `fact_2_학습.ipynb`의 값과 **똑같이** 맞춰 주세요.
- 테스트가 끝나면 꼭 Stop Session (GPU 2개라 시간이 빨리 닳을 수 있어요, 캐글 프로필에서 확인).
- 이 노트북은 캐글에서 아직 실행해 보지 못한 상태예요. 오류가 나면 로그를 알려 주세요.

In [ ]:
!pip -q install fastapi uvicorn nest_asyncio httpx
!pip -q install unsloth
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
import os, secrets, torch
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
HF_TOKEN = sec.get_secret("HF_TOKEN")
try:
    KEY = sec.get_secret("MODEL_KEY")          # Secrets에 넣어 두면 매번 같은 비밀번호
except Exception:
    KEY = secrets.token_urlsafe(24)            # 없으면 이번에만 쓰는 새 비밀번호

MODEL_NAME = "unsloth/gemma-4-E4B-it"   # ← fact_2_학습.ipynb 의 MODEL_NAME 과 똑같이!
LORA_REPO  = "lagem/LAGO_AI_MODELS"
LORA_SUB   = "fonte"
N_GPU = max(1, torch.cuda.device_count())
print("GPU 개수:", N_GPU)

In [ ]:
%%writefile worker.py
import os, sys, secrets, threading, torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn
PORT = int(sys.argv[1]); KEY = os.environ["MODEL_KEY"]; HF_TOKEN = os.environ["HF_TOKEN"]
MODEL_NAME = os.environ["MODEL_NAME"]; LORA_REPO = os.environ["LORA_REPO"]; LORA_SUB = os.environ["LORA_SUB"]
from unsloth import FastModel
from peft import PeftModel
model, tokenizer = FastModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=2048, load_in_4bit=True, token=HF_TOKEN)
model = PeftModel.from_pretrained(model, LORA_REPO, subfolder=LORA_SUB, token=HF_TOKEN)
FastModel.for_inference(model)
tok = getattr(tokenizer, "tokenizer", tokenizer)
app = FastAPI(); lock = threading.Lock()
class Req(BaseModel):
    system: str
    user: str
    max_new_tokens: int = 400
@app.get("/health")
def health(): return {"ok": True}
@app.post("/generate")
def generate(r: Req, x_key: str = Header(default="")):
    if not secrets.compare_digest(x_key, KEY): raise HTTPException(401, "bad key")
    msgs = [{"role": "system", "content": r.system}, {"role": "user", "content": r.user}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with lock, torch.no_grad():
        out = model.generate(**ids, max_new_tokens=min(r.max_new_tokens, 600), do_sample=True, temperature=0.2, top_p=0.9)
    return {"answer": tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()}
print("worker ready", PORT, flush=True)
uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

In [ ]:
import subprocess, time, requests, os
workers = []   # (포트, 프로세스)
for i in range(N_GPU):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), MODEL_KEY=KEY, HF_TOKEN=HF_TOKEN, MODEL_NAME=MODEL_NAME, LORA_REPO=LORA_REPO, LORA_SUB=LORA_SUB)
    port = 8101 + i
    p = subprocess.Popen(["python", "worker.py", str(port)], env=env, stdout=open(f"worker{i}.log", "w"), stderr=subprocess.STDOUT)
    workers.append((port, p))
# 모델 올리는 데 몇 분 걸려요 (GPU마다 동시에 올라가요)
for port, p in workers:
    for _ in range(120):
        try:
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).ok: break
        except Exception: pass
        if p.poll() is not None: raise RuntimeError(f"일꾼 {port}이 꺼졌어요. worker{port-8101}.log 를 확인하세요")
        time.sleep(5)
    else:
        raise RuntimeError(f"일꾼 {port}이 시간 안에 안 켜졌어요")
print("모델 준비 완료 · 일꾼", len(workers), "개")

In [ ]:
# 문 앞 안내 서버: 놀고 있는 일꾼에게 질문을 넘겨 줘요
import threading, httpx, uvicorn, nest_asyncio
from fastapi import FastAPI, Header, HTTPException, Request
nest_asyncio.apply()
gate = FastAPI()
busy = {port: 0 for port, _ in workers}
mu = threading.Lock()

@gate.get("/health")
def health():
    return {"ok": True, "workers": len(workers)}

@gate.post("/generate")
async def generate(req: Request, x_key: str = Header(default="")):
    if not secrets.compare_digest(x_key, KEY):
        raise HTTPException(401, "bad key")
    body = await req.json()
    with mu:
        port = min(busy, key=busy.get); busy[port] += 1
    try:
        async with httpx.AsyncClient(timeout=120) as c:
            r = await c.post(f"http://127.0.0.1:{port}/generate", json=body, headers={"x-key": KEY})
        if r.status_code != 200:
            raise HTTPException(502, "worker error")
        return r.json()
    finally:
        with mu: busy[port] -= 1

threading.Thread(target=lambda: uvicorn.run(gate, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(3)

In [ ]:
import re
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
URL = None
for line in tun.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0); break
print("=" * 60)
print("관리자 페이지 → 키와 설정에 아래 두 값을 넣으세요")
print("모델 서버 주소 (MODEL_URL):", URL)
print("모델 서버 비밀번호 (MODEL_KEY):", KEY)
print("=" * 60)
print("건강 확인:", requests.get(URL + "/health", timeout=30).text)

In [ ]:
# 이 셀을 계속 켜 두세요 (멈추면 사이트에서 답이 안 와요)
import time
while True:
    time.sleep(300)
    print("살아 있어요", time.strftime("%H:%M:%S"), "· 처리 중:", dict(busy))